# Aula 4, Bag of Words

Notebook executável que acompanha a aula [04-bag-of-words.md](../../lessons/modulo-03-fundamentos-nlp/04-bag-of-words.md).

## O que vamos fazer aqui

Aqui damos o salto do texto para os números. O Bag of Words representa cada documento
por um vetor de contagens de palavras, ignorando a ordem. Vamos montar a matriz
documento-termo das perguntas e medir a similaridade do cosseno entre elas.

O resultado vai trazer uma surpresa instrutiva, que expõe a maior fraqueza da
contagem crua e motiva o TF-IDF da próxima aula. Python puro.

## Matriz documento-termo

Primeiro construímos o vocabulário, em ordem fixa, e depois transformamos cada
documento em um vetor com a contagem de cada palavra. Empilhar esses vetores forma a
matriz documento-termo.

In [ ]:
import re
import math

documentos = [
    "como faço a derivada de uma função",
    "qual a regra da cadeia na derivada",
    "como resolvo um sistema linear com matrizes",
    "como declaro uma função em python",
]


def tokenizar(texto):
    return re.findall(r"\w+", texto.lower(), re.UNICODE)


vocabulario = sorted({palavra for doc in documentos for palavra in tokenizar(doc)})
indice = {palavra: i for i, palavra in enumerate(vocabulario)}


def vetor_bow(texto):
    vetor = [0] * len(vocabulario)
    for palavra in tokenizar(texto):
        if palavra in indice:
            vetor[indice[palavra]] += 1
    return vetor


matriz = [vetor_bow(doc) for doc in documentos]
print("Tamanho do vocabulário:", len(vocabulario))

Cada documento virou um vetor do tamanho do vocabulário. Note que doc0 e
doc1 são de cálculo, doc2 é de álgebra e doc3 é de programação. Vamos medir o quanto
eles se parecem.

## Similaridade do cosseno

A similaridade do cosseno mede o ângulo entre dois vetores, indo de 0 (nada em comum)
a 1 (mesma direção). Vamos comparar a primeira pergunta de cálculo com as demais.

In [ ]:
def cosseno(a, b):
    produto = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb_ = math.sqrt(sum(y * y for y in b))
    return produto / (na * nb_) if na and nb_ else 0.0


print("doc0 e doc1 (cálculo x cálculo):    ", round(cosseno(matriz[0], matriz[1]), 3))
print("doc0 e doc2 (cálculo x álgebra):    ", round(cosseno(matriz[0], matriz[2]), 3))
print("doc0 e doc3 (cálculo x programação):", round(cosseno(matriz[0], matriz[3]), 3))

Surpresa: a pergunta de cálculo (doc0) fica mais parecida com a de
programação (doc3, cosseno perto de 0,46) do que com a outra de cálculo (doc1, perto
de 0,29). O motivo é que doc0 e doc3 compartilham palavras comuns como como, uma e
função, enquanto doc0 e doc1 só têm em comum o termo derivada. Com a contagem crua, as
palavras genéricas pesam tanto quanto as de conteúdo e mandam no resultado.

Esse é o calcanhar de aquiles do Bag of Words puro, e é exatamente o que o TF-IDF, na
próxima aula, vem corrigir, rebaixando as palavras comuns. Para o projeto, construa
uma máquina de similaridade que devolve a pergunta mais parecida e encontre um caso em
que ela erra por falta de noção de sinônimos.